# 03b Bronze securities and prices
Ingests securities.csv and prices.csv from the landing Volume using Auto Loader.

In [0]:

from pyspark.sql import functions as F

source = "/Volumes/workspace/bronze/landing/"
chk = "/Volumes/workspace/bronze/checkpoints/securities"

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", True)
    .option("pathGlobFilter", "securities.csv")
    .option("cloudFiles.schemaLocation", chk + "/schema")
    .load(source)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    stream.writeStream
    .option("checkpointLocation", chk + "/state")
    .trigger(availableNow=True)
    .toTable("bronze.securities")
)

In [0]:
chk = "/Volumes/workspace/bronze/checkpoints/prices"

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", True)
    .option("pathGlobFilter", "prices.csv")
    .option("cloudFiles.schemaLocation", chk + "/schema")
    .load(source)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    stream.writeStream
    .option("checkpointLocation", chk + "/state")
    .trigger(availableNow=True)
    .toTable("bronze.prices")
)

In [0]:
%sql
SELECT 'securities' AS tbl, COUNT(*) AS row_count FROM bronze.securities
UNION ALL
SELECT 'prices', COUNT(*) FROM bronze.prices